In [10]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    classification_report,
    roc_auc_score

)
from tqdm import tqdm

In [11]:
# Download the preprocessed data
X_train = pd.read_csv("../clean_data/X_train.csv")
X_test = pd.read_csv("../clean_data/X_test.csv")
y_train = pd.read_csv("../clean_data/y_train.csv")
y_test = pd.read_csv("../clean_data/y_test.csv")

In [12]:
# Encodage de la variaable mois en variables sin et cos pour capturer la cyclicité
X_train["Month_sin"] = np.sin(2 * np.pi * X_train["Month"] / 12)
X_train["Month_cos"] = np.cos(2 * np.pi * X_train["Month"] / 12)

X_test["Month_sin"] = np.sin(2 * np.pi * X_test["Month"] / 12)
X_test["Month_cos"] = np.cos(2 * np.pi * X_test["Month"] / 12)

# Encodage de la variable "Day" en variables sin et cos pour capturer la cyclicité
X_train["Day_sin"] = np.sin(2 * np.pi * X_train["Day"] / 31)
X_train["Day_cos"] = np.cos(2 * np.pi * X_train["Day"] / 31)

X_test["Day_sin"] = np.sin(2 * np.pi * X_test["Day"] / 31)
X_test["Day_cos"] = np.cos(2 * np.pi * X_test["Day"] / 31)

# Sppression de la variable "Month" et "Day" après l'encodage
X_train = X_train.drop(columns=["Month", "Day"])
X_test = X_test.drop(columns=["Month", "Day"])

X_train.head()

,Transaction_Amount (in Million),Distance_From_Home,Account_Balance (in Million),Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,...,Customer_Home_Location_Islamabad,Customer_Home_Location_Karachi,Customer_Home_Location_Lahore,Customer_Home_Location_Multan,Card_Type_Credit,Card_Type_Debit,Month_sin,Month_cos,Day_sin,Day_cos
0,1.0,212.0,6.0,6.0,1.0,2.0,5.0,1,1,2.0,...,0,0,1,0,1,0,0.866025,-0.5,-0.998717,-0.050649
1,2.0,170.0,32.0,6.0,22.0,2.0,5.0,1,0,0.0,...,0,0,0,0,0,1,0.866025,-0.5,0.848644,0.528964
2,4.0,49.0,39.0,2.0,3.0,4.0,7.0,0,1,1.0,...,0,0,0,0,0,1,0.866025,-0.5,-0.101168,-0.994869
3,4.0,212.0,10.0,2.0,8.0,3.0,3.0,1,1,1.0,...,0,1,0,0,1,0,0.866025,-0.5,-0.988468,0.151428
4,9.0,507.0,30.0,1.0,21.0,5.0,3.0,0,1,0.0,...,0,0,1,0,0,1,0.866025,-0.5,0.988468,0.151428


In [13]:
print("Entraînement du modèle...")
   # Création du modèle
with tqdm(total=1) as pbar:
    model = LogisticRegression(
        random_state=42,
        class_weight="balanced",
        max_iter=1000
    )

    # Entraînement
    model.fit(X_train, y_train)

    pbar.update(1)

print("Entraînement terminé !")

Entraînement du modèle...


  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\tapon\anaconda3\envs\fraud310\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\tapon\anaconda3\envs\fraud310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
100%|██████████| 1/1 [00:03<00:00,  3.43s/it]

Entraînement terminé !


In [14]:
# Prédictions
y_pred = model.predict(X_test)

In [15]:
# Evaluation
print("=" * 50)
print("Logistic Regression")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, pos_label=1):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, pos_label=1):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred, pos_label=1):.4f}")

print("\nMatrice de confusion")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de classification")
print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))


Logistic Regression
Accuracy  : 0.6106
Precision : 0.0622
Recall    : 0.4990
F1-score  : 0.1106

Matrice de confusion
[[5864 3651]
 [ 243  242]]

Rapport de classification
              precision    recall  f1-score   support

      Normal       0.96      0.62      0.75      9515
       Fraud       0.06      0.50      0.11       485

    accuracy                           0.61     10000
   macro avg       0.51      0.56      0.43     10000
weighted avg       0.92      0.61      0.72     10000

